# 📚 Système de Recommandation NLP
## Recommander des Livres, Audiobooks et Podcasts basé sur les Préférences Utilisateur

Ce notebook implémente un système de recommandation intelligent utilisant NLP pour proposer du contenu personnalisé à l'utilisateur en fonction de ses réponses au formulaire de préférences.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Pour l'interface interactive
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ Toutes les bibliothèques importées avec succès!")

## 1️⃣ Créer et Charger les Données

In [ ]:
# Créer des données d'exemple pour les livres
books_data = {
    'id': ['L1', 'L2', 'L3', 'L4', 'L5'],
    'type': ['Livre', 'Livre', 'Livre', 'Livre', 'Livre'],
    'titre': ['Le Seigneur des Anneaux', 'Orgueil et Préjugés', 'Le Code Da Vinci', '1984', 'Dune'],
    'auteur': ['J.R.R. Tolkien', 'Jane Austen', 'Dan Brown', 'George Orwell', 'Frank Herbert'],
    'genre': ['Fantasy', 'Romance', 'Thriller', 'Science-Fiction', 'Science-Fiction'],
    'description': [
        'Une épopée fantasy épique avec des créatures magiques, des aventures héroïques et un monde riche',
        'Un roman de romance classique avec de l\'intrigue, des conflits sociaux et une histoire d\'amour passionnante',
        'Un thriller mystérieux combinant art, histoire et codes secrets avec beaucoup de suspense',
        'Un roman dystopique sombre explorant la manipulation politique, la surveillance et la liberté',
        'Une saga science-fiction complexe avec planètes exotiques, politique intergalactique et écologie'
    ],
    'duree_heures': [25, 12, 17, 15, 22]
}

# Créer des données d'exemple pour les audiobooks
audiobooks_data = {
    'id': ['A1', 'A2', 'A3', 'A4', 'A5'],
    'type': ['Audiobook', 'Audiobook', 'Audiobook', 'Audiobook', 'Audiobook'],
    'titre': ['Fondation', 'Les Misérables', 'Harry Potter', 'Neuromancien', 'Le Hobbit'],
    'auteur': ['Isaac Asimov', 'Victor Hugo', 'J.K. Rowling', 'William Gibson', 'J.R.R. Tolkien'],
    'genre': ['Science-Fiction', 'Historique', 'Fantasy', 'Cyberpunk', 'Fantasy'],
    'description': [
        'Une série science-fiction épique sur l\'effondrement d\'un empire galactique et sa reconstruction',
        'Un classique historique épique explorant la justice, la rédemption et la révolution française',
        'Une série fantastique jeunesse suivant un jeune sorcier dans un monde magique rempli d\'aventures',
        'Un roman cyberpunk révolutionnaire avec intelligence artificielle, réalité virtuelle et hacking',
        'Une aventure fantasy épique suivant un petit hobbit dans une quête extraordinaire'
    ],
    'duree_heures': [30, 45, 140, 10, 12]
}

# Créer des données d'exemple pour les podcasts
podcasts_data = {
    'id': ['P1', 'P2', 'P3', 'P4', 'P5'],
    'type': ['Podcast', 'Podcast', 'Podcast', 'Podcast', 'Podcast'],
    'titre': ['Histoires Mystérieuses', 'Science Expliquée', 'Interviews Créatives', 'Voyage dans le Temps', 'Thriller Sonore'],
    'animateur': ['Marie Dupont', 'Dr. Jean Michel', 'Sophie Lefevre', 'Pierre Arnaud', 'Collectif Audio'],
    'genre': ['Mystère', 'Éducatif', 'Entretien', 'Histoire', 'Thriller'],
    'description': [
        'Des histoires fascinantes et mystérieuses qui explorent le paranormal et les énigmes non résolues',
        'Des épisodes éducatifs expliquant des concepts scientifiques complexes de manière simple et engageante',
        'Des interviews inspirantes avec des créatifs, artistes et entrepreneurs partageant leurs histoires uniques',
        'Voyage fascinant à travers les événements historiques majeurs qui ont façonné le monde',
        'Des drames audio captivants avec suspense, intrigues et personnages complexes'
    ],
    'duree_minutes': [45, 35, 60, 50, 55]
}

# Créer DataFrames
df_books = pd.DataFrame(books_data)
df_audiobooks = pd.DataFrame(audiobooks_data)
df_podcasts = pd.DataFrame(podcasts_data)

# Combiner tous les datasets
df_content = pd.concat([df_books, df_audiobooks, df_podcasts], ignore_index=True)

print("✅ Données chargées avec succès!")
print(f"\n📊 Total: {len(df_content)} contenus")
print(f"   - {len(df_books)} Livres")
print(f"   - {len(df_audiobooks)} Audiobooks")
print(f"   - {len(df_podcasts)} Podcasts")
print("\n📚 Aperçu des données:")
df_content[['id', 'type', 'titre', 'genre']]

## 2️⃣ Prétraitement et Nettoyage du Texte

In [ ]:
def clean_text(text):
    """
    Nettoie et prétraite le texte
    """
    # Convertir en minuscules
    text = str(text).lower()
    # Supprimer les accents (optionnel)
    text = re.sub(r'[àâä]', 'a', text)
    text = re.sub(r'[éèêë]', 'e', text)
    text = re.sub(r'[îï]', 'i', text)
    text = re.sub(r'[ôö]', 'o', text)
    text = re.sub(r'[ûü]', 'u', text)
    # Supprimer les caractères spéciaux
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Supprimer les espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Appliquer le nettoyage aux données
df_content['description_cleaned'] = df_content['description'].apply(clean_text)
df_content['titre_cleaned'] = df_content['titre'].apply(clean_text)
df_content['genre_cleaned'] = df_content['genre'].apply(clean_text)

# Combiner les champs textuels pour une meilleure analyse
df_content['text_combined'] = (
    df_content['titre_cleaned'] + ' ' +
    df_content['genre_cleaned'] + ' ' +
    df_content['description_cleaned']
)

print("✅ Texte prétraité et nettoyé!")
print("\n📝 Exemple de texte combiné nettoyé:")
print(df_content['text_combined'].iloc[0][:200] + "...")

## 3️⃣ Créer les Embeddings NLP avec TF-IDF

In [ ]:
# Créer le vectoriseur TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    max_features=100,  # Nombre maximum de features
    min_df=1,  # Minimum document frequency
    max_df=0.9,  # Maximum document frequency
    ngram_range=(1, 2)  # Unigrammes et bigrammes
)

# Générer les embeddings TF-IDF pour tous les contenus
tfidf_matrix = tfidf_vectorizer.fit_transform(df_content['text_combined'])

print("✅ Embeddings TF-IDF créés!")
print(f"\n📊 Matrice TF-IDF:")
print(f"   - Nombre d'items: {tfidf_matrix.shape[0]}")
print(f"   - Nombre de features: {tfidf_matrix.shape[1]}")
print(f"   - Densité: {(tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))*100:.2f}%")

# Afficher les top features (mots/termes les plus importants)
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\n🔤 Exemples de features (top 20):")
print(feature_names[:20])

## 4️⃣ Formulaire de Préférences Utilisateur

In [ ]:
# Créer le formulaire de préférences
output = widgets.Output()

# Titres et descriptions
title_label = widgets.HTML("<h3>📋 Formulaire de Préférences Personnelles</h3>")
description_label = widgets.HTML("<p>Remplissez ce formulaire pour recevoir des recommandations de contenu personnalisées</p>")

# Sélection de genres
genre_checkbox = widgets.SelectMultiple(
    options=['Fantasy', 'Science-Fiction', 'Romance', 'Thriller', 'Historique', 'Mystère', 'Éducatif', 'Entretien', 'Cyberpunk'],
    description='Genres préférés:',
    rows=5
)

# Intérêts et mots-clés
interests_text = widgets.Textarea(
    description='Descriptions d\'intérêts:',
    placeholder='Ex: aventure, magie, espionnage, futur dystopique...',
    rows=3
)

# Sélection du type de contenu
content_type_checkbox = widgets.SelectMultiple(
    options=['Livre', 'Audiobook', 'Podcast'],
    description='Type de contenu:',
    rows=3,
    value=['Livre', 'Audiobook']
)

# Durée préférée
duration_slider = widgets.IntSlider(
    description='Durée max (heures):',
    value=30,
    min=1,
    max=200,
    step=5
)

# Bouton de soumission
submit_button = widgets.Button(
    description='Obtenir Recommandations',
    button_style='success'
)

# Résultat
result_output = widgets.Output()

def on_submit_clicked(b):
    clear_output(wait=True)
    with result_output:
        print("⏳ Analyse en cours...")

# Ajouter un listener au bouton
submit_button.on_click(on_submit_clicked)

# Créer la mise en page
form_layout = widgets.VBox([
    title_label,
    description_label,
    genre_checkbox,
    interests_text,
    content_type_checkbox,
    duration_slider,
    submit_button,
    result_output
])

display(form_layout)

## 5️⃣ Calcul de Similarité et Recommandation

In [ ]:
def get_recommendations(genres, interests, content_types, max_duration, num_recommendations=5):
    """
    Fonction pour obtenir les recommandations basées sur les préférences utilisateur
    
    Args:
        genres: liste des genres sélectionnés
        interests: texte des intérêts utilisateur
        content_types: liste des types de contenu
        max_duration: durée maximale en heures
        num_recommendations: nombre de recommandations à retourner
    
    Returns:
        DataFrame avec les recommandations triées par score de similarité
    """
    
    # Créer le profil utilisateur
    user_profile_text = ' '.join(genres) + ' ' + interests
    user_profile_text = clean_text(user_profile_text)
    
    # Vectoriser le profil utilisateur
    user_vector = tfidf_vectorizer.transform([user_profile_text])
    
    # Calculer la similarité cosinus avec tous les contenus
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix)[0]
    
    # Ajouter les scores au DataFrame
    df_content['similarity_score'] = similarity_scores
    
    # Filtrer par type de contenu
    df_filtered = df_content[df_content['type'].isin(content_types)].copy()
    
    # Filtrer par durée
    # Pour les livres et audiobooks (colonne 'duree_heures')
    # Pour les podcasts (colonne 'duree_minutes')
    def check_duration(row):
        if row['type'] == 'Podcast':
            return (row['duree_minutes'] / 60) <= max_duration
        else:
            return row['duree_heures'] <= max_duration
    
    df_filtered = df_filtered[df_filtered.apply(check_duration, axis=1)]
    
    # Filtrer par genre si au moins un genre est sélectionné
    if genres:
        df_filtered = df_filtered[df_filtered['genre'].isin(genres)]
    
    # Trier par score de similarité (décroissant)
    df_filtered = df_filtered.sort_values('similarity_score', ascending=False)
    
    # Retourner les top N recommandations
    return df_filtered[['id', 'type', 'titre', 'auteur', 'genre', 'similarity_score']].head(num_recommendations)

# Test avec des préférences d'exemple
print("🧪 Test de la fonction de recommandation:")
print("=" * 80)

test_recommendations = get_recommendations(
    genres=['Fantasy', 'Science-Fiction'],
    interests='aventure épique, mondes magiques, créatures fantastiques',
    content_types=['Livre', 'Audiobook', 'Podcast'],
    max_duration=50,
    num_recommendations=5
)

print("\n📚 Recommandations de test:")
print(test_recommendations.to_string())

## 6️⃣ Interface Interactive Complète

In [ ]:
# Créer une interface interactive complète
class RecommendationEngine:
    def __init__(self):
        self.output_area = widgets.Output()
        self.recommendations_df = None
    
    def create_form(self):
        """Créer le formulaire interactif"""
        
        # Titre
        title = widgets.HTML("<h2>🎯 Système de Recommandation de Contenu</h2>")
        subtitle = widgets.HTML("<p style='font-size: 14px; color: #666;'>Découvrez des livres, audiobooks et podcasts adaptés à vos préférences!</p>")
        
        # Genres
        self.genres = widgets.SelectMultiple(
            options=['Fantasy', 'Science-Fiction', 'Romance', 'Thriller', 'Historique', 'Mystère', 'Éducatif', 'Entretien', 'Cyberpunk'],
            description='📖 Genres:',
            rows=4,
            style={'description_width': '120px'}
        )
        
        # Intérêts
        self.interests = widgets.Textarea(
            value='',
            placeholder='Décrivez vos intérêts, thèmes favoris (ex: magie, espionnage, futur, amour...)',
            description='💭 Intérêts:',
            rows=3,
            style={'description_width': '120px'}
        )
        
        # Type de contenu
        self.content_type = widgets.CheckboxGroup(
            options=['Livre', 'Audiobook', 'Podcast'],
            description='📚 Type:',
            style={'description_width': '120px'}
        )
        
        # Durée
        self.duration = widgets.IntSlider(
            value=50,
            min=1,
            max=200,
            step=5,
            description='⏱️ Durée max (h):',
            style={'description_width': '120px'}
        )
        
        # Nombre de recommandations
        self.num_recs = widgets.IntSlider(
            value=5,
            min=1,
            max=15,
            step=1,
            description='🔢 Nombre:',
            style={'description_width': '120px'}
        )
        
        # Bouton
        self.button = widgets.Button(
            description='🚀 Obtenir Recommandations',
            button_style='success',
            tooltip='Cliquez pour générer vos recommandations'
        )
        self.button.on_click(self.on_button_clicked)
        
        # Mise en page
        form_items = widgets.VBox([
            title,
            subtitle,
            widgets.HTML("<hr style='margin: 20px 0;'>"),
            self.genres,
            self.interests,
            self.content_type,
            self.duration,
            self.num_recs,
            self.button,
            self.output_area
        ])
        
        return form_items
    
    def on_button_clicked(self, b):
        """Traiter le clic sur le bouton"""
        self.output_area.clear_output(wait=False)
        
        with self.output_area:
            # Vérifier les sélections
            if not self.genres.value and not self.interests.value:
                print("⚠️ Veuillez sélectionner au moins un genre ou entrer vos intérêts!")
                return
            
            if not self.content_type.value:
                print("⚠️ Veuillez sélectionner au moins un type de contenu!")
                return
            
            print("⏳ Analyse en cours...")
            print("=" * 80)
            
            # Obtenir les recommandations
            recommendations = get_recommendations(
                genres=list(self.genres.value),
                interests=self.interests.value,
                content_types=list(self.content_type.value),
                max_duration=self.duration.value,
                num_recommendations=self.num_recs.value
            )
            
            if len(recommendations) == 0:
                print("❌ Aucune recommandation trouvée avec ces critères.")
                print("Essayez de modifier vos filtres (durée, genres, etc.)")
                return
            
            # Afficher les résultats
            print(f"\n✨ {len(recommendations)} Recommandation(s) trouvée(s) pour vous!\n")
            
            for idx, (_, row) in enumerate(recommendations.iterrows(), 1):
                score_percent = int(row['similarity_score'] * 100)
                print(f"#{idx} | {row['titre'].upper()}")
                print(f"    Type: {row['type']} | Genre: {row['genre']}")
                print(f"    Auteur: {row['auteur']}")
                print(f"    Score de correspondance: {score_percent}%")
                print()
            
            self.recommendations_df = recommendations

# Créer et afficher le moteur de recommandation
engine = RecommendationEngine()
display(engine.create_form())

## 7️⃣ Analyse et Visualisation des Résultats

In [ ]:
def visualize_similarities(user_interests, top_n=10):
    """
    Visualiser la distribution des scores de similarité
    """
    # Nettoyer et vectoriser les intérêts
    user_text = clean_text(user_interests)
    user_vector = tfidf_vectorizer.transform([user_text])
    
    # Calculer les scores
    scores = cosine_similarity(user_vector, tfidf_matrix)[0]
    df_temp = df_content.copy()
    df_temp['score'] = scores
    df_temp = df_temp.sort_values('score', ascending=False).head(top_n)
    
    # Créer la visualisation
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = ['#2ecc71' if s > 0.5 else '#3498db' if s > 0.3 else '#95a5a6' for s in df_temp['score']]
    
    bars = ax.barh(range(len(df_temp)), df_temp['score'], color=colors)
    ax.set_yticks(range(len(df_temp)))
    ax.set_yticklabels(df_temp['titre'])
    ax.set_xlabel('Score de Similarité', fontsize=12, fontweight='bold')
    ax.set_title('Top 10 Contenus les Plus Similaires à vos Intérêts', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 1)
    
    # Ajouter les valeurs sur les barres
    for i, (bar, score) in enumerate(zip(bars, df_temp['score'])):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
               f'{score:.2%}', ha='left', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def analyze_dataset():
    """
    Analyser et afficher des statistiques sur la base de données
    """
    print("📊 STATISTIQUES DE LA BASE DE DONNÉES")
    print("=" * 80)
    
    print(f"\n📈 Répartition par type:")
    type_counts = df_content['type'].value_counts()
    for content_type, count in type_counts.items():
        print(f"   • {content_type}: {count} contenus")
    
    print(f"\n🎭 Répartition par genre:")
    genre_counts = df_content['genre'].value_counts()
    for genre, count in genre_counts.items():
        print(f"   • {genre}: {count} contenus")
    
    print(f"\n⏱️ Statistiques de durée:")
    print(f"   • Livres/Audiobooks: {df_content[df_content['type'].isin(['Livre', 'Audiobook'])]['duree_heures'].min():.1f}h - {df_content[df_content['type'].isin(['Livre', 'Audiobook'])]['duree_heures'].max():.1f}h")
    print(f"   • Podcasts: {df_content[df_content['type'] == 'Podcast']['duree_minutes'].min():.0f}min - {df_content[df_content['type'] == 'Podcast']['duree_minutes'].max():.0f}min")

# Afficher l'analyse
analyze_dataset()

print("\n" + "=" * 80)
print("📝 Pour utiliser le système:")
print("   1. Remplissez le formulaire ci-dessus")
print("   2. Cliquez sur 'Obtenir Recommandations'")
print("   3. Les résultats s'afficheront ci-dessus avec les scores de correspondance")
print("=" * 80)

## 8️⃣ Exemples d'Utilisation et Tests

In [ ]:
print("=" * 80)
print("EXEMPLE 1: Amateur de Fantasy & Aventure")
print("=" * 80)

example1_recs = get_recommendations(
    genres=['Fantasy'],
    interests='aventure épique, dragon, magie, quête héroïque',
    content_types=['Livre', 'Audiobook', 'Podcast'],
    max_duration=40,
    num_recommendations=5
)

for idx, (_, row) in enumerate(example1_recs.iterrows(), 1):
    score_percent = int(row['similarity_score'] * 100)
    print(f"#{idx} - {row['titre']} ({row['type']}) | Score: {score_percent}%")

print("\n" + "=" * 80)
print("EXEMPLE 2: Fan de Science-Fiction & Cyberpunk")
print("=" * 80)

example2_recs = get_recommendations(
    genres=['Science-Fiction', 'Cyberpunk'],
    interests='futur dystopique, intelligence artificielle, hackers, technologie avancée',
    content_types=['Livre', 'Audiobook'],
    max_duration=50,
    num_recommendations=5
)

for idx, (_, row) in enumerate(example2_recs.iterrows(), 1):
    score_percent = int(row['similarity_score'] * 100)
    print(f"#{idx} - {row['titre']} ({row['type']}) | Score: {score_percent}%")

print("\n" + "=" * 80)
print("EXEMPLE 3: Préférence pour les Podcasts")
print("=" * 80)

example3_recs = get_recommendations(
    genres=['Histoire', 'Éducatif'],
    interests='histoires intéressantes, apprentissage, mystères historiques',
    content_types=['Podcast'],
    max_duration=200,
    num_recommendations=5
)

for idx, (_, row) in enumerate(example3_recs.iterrows(), 1):
    score_percent = int(row['similarity_score'] * 100)
    print(f"#{idx} - {row['titre']} ({row['genre']}) | Score: {score_percent}%")

print("\n" + "=" * 80)
print("✅ Le système de recommandation NLP est prêt!")
print("=" * 80)

## 9️⃣ Documentation & Guide d'Intégration

### 🔧 Architecture du Système

**1. TF-IDF Vectorization**
- Convertit les descriptions textuelles en vecteurs numériques
- Identifie les termes les plus importants et distincts
- Crée une représentation 100-dimensionnelle de chaque contenu

**2. Cosine Similarity**
- Mesure la similarité entre le profil utilisateur et chaque contenu
- Score entre 0 (aucune similarité) et 1 (similarité totale)
- Basé sur l'angle entre les vecteurs

**3. Filtrage Multi-critères**
- Filtre par type de contenu (Livre, Audiobook, Podcast)
- Filtre par genre sélectionné
- Filtre par durée maximale
- Trie par score de similarité décroissant

### 📌 Comment Intégrer dans votre Application

Pour intégrer ce système dans votre appli:

```python
# 1. Importer la fonction
from nlp_recommendations.ipynb import get_recommendations

# 2. Récupérer les réponses du formulaire utilisateur
user_genres = ['Fantasy', 'Science-Fiction']
user_interests = 'aventure, magie, futur'
user_content_types = ['Livre', 'Audiobook']
user_duration = 50

# 3. Obtenir les recommandations
recommendations = get_recommendations(
    genres=user_genres,
    interests=user_interests,
    content_types=user_content_types,
    max_duration=user_duration,
    num_recommendations=5
)

# 4. Afficher/retourner les résultats
for item in recommendations.iterrows():
    print(f"{item['titre']} - Score: {item['similarity_score']:.2%}")
```

### 🚀 Améliorations Futures

- Intégrer des embeddings Word2Vec ou BERT (meilleure compréhension sémantique)
- Ajouter un système de recommandation collaboratif (CF)
- Mémoriser les préférences utilisateur pour améliorer les recommandations
- Ajouter des critères de notation utilisateur
- Implémenter un système d'apprentissage actif

### 📚 Technologies Utilisées

- **Pandas**: Manipulation des données
- **Scikit-learn**: TF-IDF et similarité cosinus
- **NumPy**: Calculs numériques
- **IPyWidgets**: Interface interactive
- **Matplotlib/Seaborn**: Visualisation

### ⚙️ Paramètres Configurables

- `max_features`: Nombre maximum de features TF-IDF (défaut: 100)
- `min_df`: Fréquence minimale des documents (défaut: 1)
- `max_df`: Fréquence maximale des documents (défaut: 0.9)
- `ngram_range`: Unigrammes et bigrammes (défaut: (1, 2))